# Projet 5 - Finance comportementale

## Question et limites

Ce projet étudie l'aversion aux pertes de la théorie des perspectives et illustre l'effet de disposition décrit par Shefrin et Statman : vendre les titres gagnants trop tôt et conserver les titres perdants trop longtemps. Les cours seuls ne révèlent pas les transactions ni les prix de revient des investisseurs; l'effet de disposition n'est donc pas mesuré empiriquement ici. Le test de portefeuille ci-dessous est une simulation stylisée, pas une estimation du comportement réel.

L'univers reprend les dix titres du Projet 4 afin de faciliter la comparaison : AAPL, MSFT, GOOG, NVDA, AMZN, 2222.SR, TSM, AVGO, META et TSLA. Il s'agit d'un univers sélectionné selon une capitalisation observée en 2026; le biais de survivance et l'absence de titres radiés limitent les conclusions. Les séances sont alignées sur les dates communes en conservant le dernier cours connu.

## Méthode

Le point de référence est le cours de clôture 63 séances auparavant. La fonction de valeur est `v(x) = x**alpha` pour les gains et `-lambda * (-x)**beta` pour les pertes. Les paramètres par défaut (`lambda=2.25`, `alpha=beta=0.88`) sont des hypothèses illustratives, non estimées sur ces titres.

La simulation « disposition » prend un gain à +5 % et tolère une baisse jusqu'à -20 %; le témoin symétrique utilise +5 % et -5 %. Après une sortie, le capital de la ligne reste en liquidités pendant une séance, puis la ligne est réouverte. Les pondérations initiales par titre restent fixes, les positions ne s'appuient que sur les cours connus à la clôture, et des frais sont déduits lors des variations de position. Le benchmark rééquilibre quotidiennement un panier équipondéré. Seuils et coûts sont des hypothèses sensibles; cette expérience ne constitue pas un conseil d'investissement.

## Références

- Kahneman, D. & Tversky, A. (1979), “Prospect Theory: An Analysis of Decision under Risk”, *Econometrica*, 47(2). [DOI: 10.2307/1914185](https://doi.org/10.2307/1914185).
- Shefrin, H. & Statman, M. (1985), “The Disposition to Sell Winners Too Early and Ride Losers Too Long: Theory and Evidence”, *The Journal of Finance*, 40(3), 777–790. [DOI: 10.1111/j.1540-6261.1985.tb05002.x](https://doi.org/10.1111/j.1540-6261.1985.tb05002.x).

In [ ]:
from pathlib import Path
import sys

racine = Path.cwd().resolve()
while not (racine / "src" / "behavioral_strategy.py").is_file() and racine != racine.parent:
    racine = racine.parent
if not (racine / "src" / "behavioral_strategy.py").is_file():
    raise FileNotFoundError("Impossible de localiser src/behavioral_strategy.py")
sys.path.insert(0, str(racine / "src"))

import pandas as pd
import plotly.graph_objects as go
from behavioral_strategy import (
    UNIVERS_TITRES,
    backtest_finance_comportementale,
    backtest_croisement_moyennes_finance_comportementale,
    valeur_prospect,
)
from metrics import metriques

TICKERS = [ticker for ticker, _ in UNIVERS_TITRES]
print(f"Univers: {', '.join(TICKERS)}")

In [ ]:
resultat, resume_titres = backtest_finance_comportementale(
    tickers=TICKERS,
    debut="2018-01-01",
    reference_window=63,
    lambda_perte=2.25,
    alpha=0.88,
    beta=0.88,
    seuil_gain=0.05,
    seuil_perte_disposition=0.20,
    seuil_perte_symetrique=0.05,
    cout=0.001,
)
print(f"Période commune: {resultat.index[0].date()} — {resultat.index[-1].date()}")
display(resume_titres.round(4))

In [ ]:
colonnes_strategies = {
    "Disposition stylisée": "strategie_disposition",
    "Contrôle symétrique": "strategie_symetrique",
    "Panier équipondéré": "benchmark_equipondere",
}
comparaison = pd.DataFrame(
    {nom: metriques(resultat[colonne]) for nom, colonne in colonnes_strategies.items()}
)
display(comparaison)

figure = go.Figure()
for nom, colonne in colonnes_strategies.items():
    valeur = (1 + resultat[colonne]).cumprod() * 100
    figure.add_trace(go.Scatter(x=valeur.index, y=valeur, name=nom))
figure.update_layout(
    title="Projet 5 - stratégies comportementales stylisées",
    yaxis_title="Valeur (base 100)",
    xaxis_title="Date",
    hovermode="x unified",
    template="plotly_white",
)
figure.show()

In [ ]:
evenements = resultat.attrs["evenements_sortie"]
if evenements.empty:
    print("Aucune sortie simulée pendant la période.")
else:
    display(
        evenements.groupby(["modele", "type_sortie"])
        .agg(
            nombre=("ticker", "size"),
            rendement_moyen_pct=("rendement_realise_pct", "mean"),
            duree_moyenne_seances=("duree_seances", "mean"),
        )
        .round(2)
    )

exemples = pd.Series(
    valeur_prospect([-0.10, 0.10]),
    index=["perte de 10 %", "gain de 10 %"],
    name="Valeur selon la fonction de référence",
)
display(exemples.to_frame())

In [ ]:
resultat_mm = backtest_croisement_moyennes_finance_comportementale(
    tickers=TICKERS,
    debut="2018-01-01",
    courte=20,
    longue=50,
    cout=0.001,
)
comparaison_mm = pd.DataFrame(
    {
        "Croisement MM 20/50": metriques(resultat_mm["strategie_mm"]),
        "Panier équipondéré": metriques(resultat_mm["benchmark_equipondere"]),
    }
)
display(comparaison_mm)

figure_mm = go.Figure()
for nom, colonne in {
    "Croisement MM 20/50": "strategie_mm",
    "Panier équipondéré": "benchmark_equipondere",
}.items():
    valeur = (1 + resultat_mm[colonne]).cumprod() * 100
    figure_mm.add_trace(go.Scatter(x=valeur.index, y=valeur, name=nom))
figure_mm.update_layout(
    title="Projet 5 - Croisement de moyennes mobiles 20/50",
    yaxis_title="Valeur (base 100)",
    xaxis_title="Date",
    hovermode="x unified",
    template="plotly_white",
)
figure_mm.show()

fichier_mm = dossier_sortie = racine / "backtest" / "projet5_finance_comportementale_mm.csv"
fichier_mm.parent.mkdir(parents=True, exist_ok=True)
resultat_mm.to_csv(fichier_mm, index=True)
print(f"Résultats MM sauvegardés dans {fichier_mm}")

In [ ]:
dossier_sortie = racine / "backtest"
dossier_sortie.mkdir(parents=True, exist_ok=True)
fichier_sortie = dossier_sortie / "projet5_finance_comportementale.csv"
resultat.to_csv(fichier_sortie, index=True)
print(f"Résultats sauvegardés dans {fichier_sortie}")